In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
parsed_docs = [doc.parse() for doc in files]

print(f"Loaded {len(parsed_docs)} documents")

Loaded 95 documents


Dividimos el texto en palabras y a partir de ahi hacemos una ventana

In [4]:
from gitsource.chunking import sliding_window
def tokenChunks(text):
    # Split text into words (tokens)
    words = text.split()
    # Create chunks using sliding window
    chunk_size = 500  # words per chunk
    step = 250  # overlap of 250 words
    chunks = sliding_window(words, size=chunk_size, step=step)

    print(f"Created {len(chunks)} chunks")
    return chunks
chunks = tokenChunks(parsed_docs[45]['content'])
chunks

Created 11 chunks


[{'start': 0,
  'content': ['In',
   'this',
   'tutorial,',
   'you',
   'will',
   'learn',
   'how',
   'to',
   'perform',
   'regression',
   'testing',
   'for',
   'LLM',
   'outputs.',
   'You',
   'can',
   'compare',
   'new',
   'and',
   'old',
   'responses',
   'after',
   'changing',
   'a',
   'prompt,',
   'model,',
   'or',
   'anything',
   'else',
   'in',
   'your',
   'system.',
   'By',
   're-running',
   'the',
   'same',
   'inputs',
   'with',
   'new',
   'parameters,',
   'you',
   'can',
   'spot',
   'any',
   'significant',
   'changes.',
   'This',
   'helps',
   'you',
   'push',
   'updates',
   'with',
   'confidence',
   'or',
   'identify',
   'issues',
   'to',
   'fix.',
   '<Info>',
   '**This',
   'example',
   'uses',
   'Evidently',
   'Cloud.**',
   "You'll",
   'run',
   'evals',
   'in',
   'Python',
   'and',
   'upload',
   'them.',
   'You',
   'can',
   'also',
   'skip',
   'the',
   'upload',
   'and',
   'view',
   'Reports',
   'lo

Ahora vamos a dividir el texto en parrafos
We use \n\s*\n regex pattern for splitting:

\n matches a newline
\s* matches zero or more whitespace characters
\n matches another newline
So \n\s*\n matches two newlines with optional whitespace between them

In [5]:
import re

text = parsed_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

print(f"Found {len(paragraphs)} paragraphs")
print(paragraphs[:3])

Found 153 paragraphs
['In this tutorial, you will learn how to perform regression testing for LLM outputs.', 'You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.', "<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>"]


Ahora hacemos el parseo por secciones
# Heading 1
## Heading 2  
### Heading 3

In [6]:
import re

# This regex matches markdown headers
# For level 2, it matches lines starting with "## "
header_pattern = r'^(#{2} )(.+)$'
pattern = re.compile(header_pattern, re.MULTILINE)
example_text = """
# Introduction

This is the intro section.

## Getting Started

Here's how to get started.

## Advanced Usage

Here are advanced topics.
""".strip()

parts = pattern.split(example_text)

print(f"Number of parts: {len(parts)}")
print(parts)

Number of parts: 7
['# Introduction\n\nThis is the intro section.\n\n', '## ', 'Getting Started', "\n\nHere's how to get started.\n\n", '## ', 'Advanced Usage', '\n\nHere are advanced topics.']


El resultado tienes multiples partes. Necesitamos recorrerlas para crear secciones

In [7]:
sections = []

for i in range(1, len(parts), 3):
    header = parts[i] + parts[i+1]
    header = header.strip()

    content = ""

    if i+2 < len(parts):
        content = parts[i+2].strip()

    if content:
        section = f'{header}\n\n{content}'
    else:
        section = header

    sections.append(section)

print(f"Found {len(sections)} sections")

for section in sections:
    print("---")
    print(section)

for section in sections:
    print(f"Section {i+1}: {section.split(chr(10))[0]}")

Found 2 sections
---
## Getting Started

Here's how to get started.
---
## Advanced Usage

Here are advanced topics.
Section 5: ## Getting Started
Section 5: ## Advanced Usage


Metemos esto en una funcion para reusarla¡

In [8]:
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    parts = pattern.split(text)

    sections = []

    for i in range(1, len(parts), 3):
        header = parts[i] + parts[i+1]
        header = header.strip()

        content = ""

        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header

        sections.append(section)

    return sections

In [10]:
sections = split_markdown_by_level(text, level=2)
print(f"Found {len(sections)} sections")
sections

Found 8 sections


['## 1. Installation and Imports\n\nInstall Evidently:\n\n```python\npip install evidently[llm] \n```\n\nImport the required modules:\n\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset\nfrom evidently.future.datasets import DataDefinition\nfrom evidently.future.datasets import Descriptor\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.metrics import *\nfrom evidently.future.tests import *\n\nfrom evidently.features.llm_judge import BinaryClassificationPromptTemplate\n```\n\nTo connect to Evidently Cloud:\n\n```python\nfrom evidently.ui.workspace.cloud import CloudWorkspace\n```\n\n**Optional.** To create monitoring panels as code:\n\n```python\nfrom evidently.ui.dashboards import DashboardPanelPlot\nfrom evidently.ui.dashboards import DashboardPanelTestSuite\nfrom evidently.ui.dashboards import DashboardPanelTestSuiteCounter\nfrom evidently.ui.das

Podemos combinar estas aproximaciones
1. Dividimos por secciones
2. Dividimos secciones grandes en parrafor
3. Aplicamos ventana deslizante a parrafors

In [11]:
from gitsource.chunking import sliding_window

# First split into sections
sections = split_markdown_by_level(text, level=2)
print(sections)
# Then slide over sections (3 sections per chunk, 1 section overlap)
chunks = sliding_window(sections, size=3, step=2)
print(chunks)
# Join sections in each chunk
combined_chunks = []
for chunk in chunks:
    chunk_text = "\n\n".join(chunk['content'])
    combined_chunks.append(chunk_text)

print(f"Created {len(combined_chunks)} chunks")
print(combined_chunks[0][:500] + "...")

['## 1. Installation and Imports\n\nInstall Evidently:\n\n```python\npip install evidently[llm] \n```\n\nImport the required modules:\n\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset\nfrom evidently.future.datasets import DataDefinition\nfrom evidently.future.datasets import Descriptor\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.metrics import *\nfrom evidently.future.tests import *\n\nfrom evidently.features.llm_judge import BinaryClassificationPromptTemplate\n```\n\nTo connect to Evidently Cloud:\n\n```python\nfrom evidently.ui.workspace.cloud import CloudWorkspace\n```\n\n**Optional.** To create monitoring panels as code:\n\n```python\nfrom evidently.ui.dashboards import DashboardPanelPlot\nfrom evidently.ui.dashboards import DashboardPanelTestSuite\nfrom evidently.ui.dashboards import DashboardPanelTestSuiteCounter\nfrom evidently.ui.das

LLM Chunking

In [12]:
chunking_instructions = """
Split the provided document into logical sections that make sense for a Q&A system.
Each section should be self-contained and cover a specific topic or concept.
Sections should be relatively large (3000-5000 characters).
""".strip()

print(chunking_instructions)

Split the provided document into logical sections that make sense for a Q&A system.
Each section should be self-contained and cover a specific topic or concept.
Sections should be relatively large (3000-5000 characters).


In [15]:
from pydantic import BaseModel
from openai import OpenAI
openai_client =  OpenAI()
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

class Section(BaseModel):
    title: str
    markdown: str

class Document(BaseModel):
    title: str
    sections: list[Section]

result = llm_structured(
    instructions=chunking_instructions,
    user_prompt=text,
    output_type=Document
)

In [16]:
result

Document(title='Regression Testing for LLM Outputs Tutorial', sections=[Section(title='Introduction to Regression Testing for LLM Outputs', markdown='In this tutorial, you will learn how to perform regression testing for large language model (LLM) outputs. By comparing new and old responses after modifying prompts, models, or other parameters, you can identify any significant changes. This process allows you to push updates confidently or pinpoint issues that need resolution.  \n\n**Example Overview:** This tutorial will utilize Evidently Cloud to run evaluations in Python. You have the option to upload results or view them locally. To facilitate a self-hosted approach, simply replace `CloudWorkspace` with `Workspace`.  \n\n### Tutorial Scope  \nBelow is a breakdown of what this tutorial will cover:  \n* **Create a toy dataset:** You will build a small Q&A dataset with expected answers and reference responses.  \n* **Get new answers:** You will simulate generating new answers for the s